# 02 — How far is the released selector from the global optimum?

Use the real Qwen activation archive, the fitted SSD cost from Notebook 01, and the released vlm-flash selector. For each greedy mask, solve the global fixed-cardinality problem at the **same realized row count**.

\[
\eta_R^\star=\max_{\|M\|_1=R}\frac{I(M)}{L(M)}.
\]

The notebook contains only that experiment and the dynamic program needed to define its oracle.

In [ ]:
from pathlib import Path
from itertools import product
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter1d

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".gitmodules").is_file())
MODEL = ROOT / "conclusion/python/results/01/latency_model.json"
TRACE = ROOT / "experiments/22_predicted_lambda_trim/results/activation_traces.npz"
UPSTREAM = ROOT / "conclusion/upstream/vlm-flash"
RESULTS = Path("oracle_results.csv")

RUN_ORACLE = False
MAX_TRACES_PER_SHAPE = 32       # 0 = all 504 traces
BUDGETS = tuple(np.arange(1, 10) / 10)
RTOL = 1e-6
PAPER_IMPL = "native"

## One measured model serves every projection.

Notebook 01 gives

\[
T_{\rm KiB}(z)=c_1z+\delta\max(s_{\rm KiB},z).
\]

If one weight row occupies \(q\) KiB, then

\[
T_{\rm row}(r)=b_1r+(b_2-b_1)\max(s,r),
\qquad
s=s_{\rm KiB}/q,\quad b_1=qc_1,\quad b_2=q(c_1+\delta).
\]

In [ ]:
model = json.loads(MODEL.read_text())
s_kib = model["s_kib"]
c1 = model["c1_ms_per_kib"]
delta = model["delta_ms_per_kib"]
s99 = model["s99_kib"]

print(f"T(z): s={s_kib:g} KiB, c1={1000*c1:.6f}, delta={1000*delta:.6f} µs/KiB")

In [ ]:
def costs(n, s, b1, b2):
    r = np.arange(n + 1, dtype=float)
    T = b1 * r + (b2 - b1) * np.maximum(s, r)
    T[0] = 0
    return T


def runs(mask):
    return np.flatnonzero(np.diff(np.r_[False, mask, False])).reshape(-1, 2)


def metrics(v, mask, T):
    length = np.diff(runs(mask), axis=1).ravel()
    return v[mask].sum(), T[length].sum()


def top_r(v, R):
    mask = np.zeros(len(v), bool)
    mask[np.argsort(-v, kind="stable")[:R]] = True
    return mask

## The last zero makes the ratio problem additive.

For a proposed ratio \(\lambda\),

\[
F_R(\lambda)=\max_{\|M\|_1=R}\{I(M)-\lambda L(M)\},
\qquad
F_R(\lambda)\ge0\iff\lambda\le\eta_R^\star.
\]

Let \(D_{z,r}\) be the best additive value among prefixes containing \(z\) zeros and \(r\) ones. Splitting at the last zero gives

\[
D_{z,r}=
\max_{0\le t\le r}
[D_{z-1,t}+P_{z+r}-P_{z+t}-\lambda T(r-t)].
\]

The two-line form turns the scan over \(t\) into one moving maximum for short runs and one prefix maximum for long runs.

In [ ]:
def bellman(v, R, lam, s, b1, b2, keep=False):
    Z = len(v) - R
    P = np.r_[0, np.cumsum(v)]
    r = np.arange(R + 1)
    prev = P[:R + 1] - lam * costs(R, s, b1, b2)
    table = np.empty((Z + 1, R + 1)) if keep else None
    if keep:
        table[0] = prev

    h = min(int(np.floor(s)), R)
    for z in range(1, Z + 1):
        prefix = P[z:z + R + 1]
        base = prev - prefix

        short = np.full(R + 1, -np.inf)
        if h:
            short[1:] = maximum_filter1d(
                base[:-1] + lam * b1 * r[:-1],
                size=h, origin=(h - 1) // 2, mode="constant", cval=-np.inf)

        long = np.full(R + 1, -np.inf)
        long[h + 1:] = np.maximum.accumulate(base + lam * b2 * r)[:R - h]

        prev = np.maximum.reduce([
            prev,
            prefix - lam * ((b2 - b1) * s + b1 * r) + short,
            prefix - lam * b2 * r + long,
        ])
        if keep:
            table[z] = prev

    return table if keep else prev[-1]

In [ ]:
def recover(v, R, lam, T, table):
    P = np.r_[0, np.cumsum(v)]
    mask = np.zeros(len(v), bool)
    r = R
    for z in range(len(v) - R, 0, -1):
        t = np.arange(r + 1)
        score = table[z - 1, :r + 1] + P[z + r] - P[z + t] - lam * T[r - t]
        t = int(np.argmax(score))
        mask[z + t:z + r] = True
        r = t
    mask[:r] = True
    return mask

In [ ]:
def solve(v, R, s, b1, b2):
    T = costs(len(v), s, b1, b2)
    initial = top_r(v, R)
    I, L = metrics(v, initial, T)
    if I == 0:
        return initial, 0

    lo, hi = I / L, I / (b2 * R)
    while hi - lo > RTOL * lo:
        lam = (lo + hi) / 2
        if bellman(v, R, lam, s, b1, b2) >= 0:
            lo = lam
        else:
            hi = lam

    table = bellman(v, R, lo, s, b1, b2, keep=True)
    mask = recover(v, R, lo, T, table)
    I, L = metrics(v, mask, T)
    return mask, I / L

### A small exhaustive check is enough.

This is not another experiment. It only checks that the accelerated Bellman value equals brute force for small masks, including a fractional boundary.

In [ ]:
v = np.random.default_rng(0).uniform(0, 4, 7)
masks = np.array(list(product([False, True], repeat=len(v))))
checks = 0

for s in (0.5, 2.5, 8):
    T = costs(len(v), s, 0.25, 1.0)
    for R in (1, 3, 7):
        feasible = masks[masks.sum(axis=1) == R]
        for lam in (0.1, 1.0, 10.0):
            exact = max(metrics(v, m, T)[0] - lam * metrics(v, m, T)[1] for m in feasible)
            np.testing.assert_allclose(bellman(v, R, lam, s, 0.25, 1.0), exact)
            checks += 1

print(f"{checks} exhaustive checks passed")

## The experiment uses real dense-model activations.

The Experiment 22 archive stores \(v_i=\operatorname{mean}|x_i|\) for real Qwen projection calls. A cap of 32 gives 128 traces; a cap of 0 uses all 504.

In [ ]:
def load_traces(path, cap):
    with np.load(path, allow_pickle=False) as archive:
        manifest = json.loads(str(archive["manifest_json"].item()))
        traces = []
        for entry in manifest["traces"]:
            trace = {k: v for k, v in entry.items() if k != "array"}
            trace["values"] = np.asarray(archive[entry["array"]], dtype=np.float32)
            traces.append(trace)

    if cap:
        selected = []
        for shape in sorted({t["shape"] for t in traces}):
            group = [t for t in traces if t["shape"] == shape]
            index = np.linspace(0, len(group) - 1, min(cap, len(group)), dtype=int)
            selected += [group[i] for i in index]
        traces = selected

    return sorted(traces, key=lambda t: t["trace_id"]), manifest


traces, manifest = load_traces(TRACE, MAX_TRACES_PER_SHAPE)
weight_bytes = {"float16": 2, "bfloat16": 2, "float32": 4}[manifest["model_dtype"]]

summary = pd.DataFrame([{"shape": t["shape"], "n": t["n"], "d": t["d"]} for t in traces])
summary = summary.groupby(["shape", "n", "d"]).size().rename("traces").reset_index()
summary["KiB/row"] = weight_bytes * summary.d / 1024
summary["boundary (rows)"] = s_kib / summary["KiB/row"]
display(summary.round(3))
print(f"{len(traces) * len(BUDGETS)} budget cases")

## Upstream greedy determines the cardinality.

Run the released selector first. If it realizes \(R_{\rm eff}\) rows, solve Global DP and Top-\(R\) at exactly \(R_{\rm eff}\).

In [ ]:
PAPER_GEOMETRY = {
    "896x128": (8, 8), "896x896": (8, 8),
    "896x4864": (8, 8), "4864x896": (12, 16),
}


def load_upstream():
    sys.path.insert(0, str((UPSTREAM / "src").resolve()))
    import torch
    from vlmflash import ChunkParams, LatencyTable, select_chunks
    table = LatencyTable({
        k: c1 * k + delta * max(s_kib, k)
        for k in range(1, int(np.ceil(s99)) + 2)
    })
    return torch, ChunkParams, select_chunks, table


def paper_mask(trace, v, R, upstream):
    torch, ChunkParams, select_chunks, table = upstream
    start, jump = PAPER_GEOMETRY[trace["shape"]]
    params = ChunkParams(start_kb=start, end_kb=s99 + 1,
                         step_kb=start, jump_cap_kb=jump)
    row_kib = weight_bytes * trace["d"] / 1024
    selection = select_chunks(
        torch.tensor(v, dtype=torch.float32), R, row_kib,
        table, params, impl=PAPER_IMPL)
    return selection.mask.cpu().numpy().astype(bool)

## Three masks answer the comparison.

For each case evaluate Global DP, upstream greedy, and Top-\(R\) with the same merged-run fitted cost.

In [ ]:
def one_case(trace, fraction, upstream):
    v = trace["values"].astype(float)
    v = v / v.mean() if v.sum() else v

    nominal = max(1, int(fraction * len(v)))
    paper = paper_mask(trace, v, nominal, upstream)
    R = int(paper.sum())
    if R == 0 or v.sum() == 0:
        return []

    row_kib = weight_bytes * trace["d"] / 1024
    s, b1, b2 = s_kib / row_kib, row_kib * c1, row_kib * (c1 + delta)
    T = costs(len(v), s, b1, b2)
    oracle, eta = solve(v, R, s, b1, b2)

    rows = []
    for name, mask in {
        "Global DP": oracle,
        "Upstream greedy": paper,
        "Top-R": top_r(v, R),
    }.items():
        I, L = metrics(v, mask, T)
        rows.append({
            "trace_id": trace["trace_id"], "shape": trace["shape"],
            "prompt": trace["input_index"], "fraction": fraction,
            "R_nominal": nominal, "R": R, "method": name,
            "retained_pct": 100 * I / v.sum(), "fitted_ms": L,
            "gap_pct": max(0, 100 * (1 - I / L / eta)),
            "chunks": len(runs(mask)),
        })
    return rows

In [ ]:
if RUN_ORACLE:
    upstream = load_upstream()
    rows = []
    for i, trace in enumerate(traces, 1):
        for fraction in BUDGETS:
            rows += one_case(trace, fraction, upstream)
        pd.DataFrame(rows).to_csv(RESULTS, index=False)
        if i % 10 == 0 or i == len(traces):
            print(f"{i}/{len(traces)} traces")

frame = pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame()
print(f"{len(frame) // 3 if len(frame) else 0}/{len(traces) * len(BUDGETS)} cases available")

## Three figures are enough.

They show the shape-level gap, its dependence on retained fraction, and one representative importance/latency trade-off.

In [ ]:
if not frame.empty:
    gaps = frame[frame.method != "Global DP"].groupby(
        ["shape", "method"]).gap_pct.median().unstack()
    ax = gaps.plot.bar(figsize=(9, 3.5), rot=0)
    ax.set(xlabel="Projection shape", ylabel="Median gap to global optimum (%)")
    plt.tight_layout()
    plt.show()

In [ ]:
if not frame.empty:
    paper = frame[frame.method == "Upstream greedy"]
    curves = paper.groupby(["fraction", "shape"]).gap_pct.median().unstack()
    ax = curves.plot(figsize=(8, 3.5), marker=".")
    ax.set(xlabel="Nominal retained fraction", ylabel="Median greedy gap (%)")
    plt.tight_layout()
    plt.show()

In [ ]:
if not frame.empty:
    case = frame[frame.fraction == 0.5].iloc[0]
    same = frame[(frame.trace_id == case.trace_id) & (frame.fraction == 0.5)]
    plt.figure(figsize=(7, 3.5))
    for _, row in same.iterrows():
        plt.scatter(row.fitted_ms, row.retained_pct, label=row.method)
    plt.xlabel("Fitted latency (ms)")
    plt.ylabel("Retained importance (%)")
    plt.legend()
    plt.tight_layout()
    plt.show()
    display(same[["method", "R", "retained_pct", "fitted_ms", "chunks", "gap_pct"]].round(4))

## Scope

Global DP is an offline optimum for the fitted two-line additive objective at the greedy mask's realized cardinality. It is not a lookup-table optimum, end-to-end quality result, or measured full-mask SSD wall time.

The implementation deliberately has no hashes, fingerprints, cache namespace, resume protocol, runtime metadata, raw-profile re-evaluation, or separate scaling engine.